# Classes and Objects in Python

Say you're writing a game with a hundred enemies on screen. Each one has its own health, position, and name, but they all behave the same way — they all know how to take damage, move, and attack. Writing a separate set of variables and functions for every single enemy would be unmanageable. What you actually want is to describe the *shape* of an enemy once, and then stamp out as many individual enemies from that description as you need. That's the whole point of classes and objects, and it's what this notebook works through.

(Separate notebooks cover inheritance, polymorphism, encapsulation, abstraction, operator overloading, and magic methods — each builds on what's here.)

## 1. What is a Class vs an Object?

**Why we need it:** without classes, if you need to track 100 dogs, you'd need 100 sets of separate variables (`dog1_name`, `dog1_breed`, `dog2_name`, `dog2_breed`, ...) and you'd have to copy-paste any "bark" logic for each one. There's no single place that says what a dog *is*.

**Use case:** anytime your program deals with many things of the same kind that share behavior but differ in data — users, orders, enemies in a game, bank accounts, points on a graph.

**How to implement:**
- Define a **class** — the blueprint. It says "anything of this type will have a name and a breed, and will be able to bark," but it doesn't describe any dog in particular yet.
- Create an **object** (an *instance*) from it by calling the class like a function: `Dog("Rex", "Labrador")`. This is no longer a description — it's one real dog in memory, with `"Rex"` and `"Labrador"` as its actual values. You can create as many objects as you want from the same class, and each keeps its own separate data while sharing the same behavior.

The code below defines the `Dog` class once, then creates two different dog objects from it.

In [1]:
class Dog:
    def __init__(self, name, breed):
        self.name = name      # instance attribute
        self.breed = breed    # instance attribute

    def bark(self):
        return f"{self.name} says Woof!"


rex = Dog("Rex", "Labrador")
milo = Dog("Milo", "Beagle")

print(type(Dog))        # the class itself is an object of type 'type'
print(type(rex))        # rex is an instance of Dog
print(rex.bark())
print(milo.bark())
print(rex.name, rex.breed)

<class 'type'>
<class '__main__.Dog'>
Rex says Woof!
Milo says Woof!
Rex Labrador


## 2. `__init__` and `self`

**Why we need it:** creating an object (`Dog("Rex", "Labrador")`) is useless if nothing actually stores `"Rex"` and `"Labrador"` onto it. Something has to run at creation time to set up the object's starting state, and that something needs a way to refer to "the specific object currently being built" so it knows where to attach that data.

**Use case:** every class that has data to set up on creation needs this — a `BankAccount` needs a starting `balance`, a `Point` needs `x`/`y`, a `User` needs a `username`. Any method that reads or changes one object's data (not just `__init__`) also needs a way to refer to that object.

**How to implement:**
- `__init__(self, ...)` — Python calls this automatically right after a new object is created, passing along whatever arguments you gave the class call. Inside it, you set `self.attribute = value` for each piece of starting data.
- `self` — the object itself, passed in automatically as the first parameter of every method (not just `__init__`). You never type it yourself when calling a method: `rex.bark()` is quietly rewritten by Python into `Dog.bark(rex)` — the object you called the method on becomes `self`. The cell below proves that by calling it both ways and getting the identical result.

In [2]:
rex.bark()          # normal way: Python calls Dog.bark(rex) implicitly
Dog.bark(rex)        # equivalent explicit form, proving `self` is just the instance

print(rex.bark() == Dog.bark(rex))

True


## 3. Instance Attributes vs Class Attributes

**Why we need it:** `self.name` alone only lets each object hold its own private data. But some data — a unit of measurement, a default setting, a running count of how many objects exist — genuinely belongs to *every* object of a type equally, not to one of them specifically. Storing it separately on every instance would be wasteful and risks them drifting out of sync.

**Use case:** shared configuration (`Rectangle.unit = "cm"`), constants tied to the class, or bookkeeping across all instances (like counting how many objects have been created).

**How to implement:**
- **Instance attribute** — assigned via `self.x = ...`, usually inside `__init__`. Each object gets its own separate copy.
- **Class attribute** — assigned directly in the class body, outside any method. Every instance can read it, and unless an instance sets its own attribute of the same name, they're all looking at the exact same value.

**Watch out for:** if a class attribute's value is mutable (like a list), every instance isn't just sharing the *value* — they're sharing the *same object*. Append to it through one instance and every other instance sees the change, because there was never more than one list to begin with. The `Team` example below deliberately breaks this way, so you can see it happen.

In [3]:
class Counter:
    total_created = 0          # class attribute: shared across all instances

    def __init__(self, name):
        self.name = name       # instance attribute: unique per instance
        Counter.total_created += 1


a = Counter("a")
b = Counter("b")
c = Counter("c")

print(f"total_created (via class): {Counter.total_created}")
print(f"total_created (via instance a): {a.total_created}")
print(f"total_created (via instance b): {b.total_created}")

# Pitfall: a mutable class attribute is shared by reference
class Team:
    members = []  # BAD: shared list across all Team instances

    def add(self, name):
        self.members.append(name)

t1 = Team()
t2 = Team()
t1.add("Alice")
print(f"t2.members also affected: {t2.members}")  # ['Alice'] -- unexpected!

total_created (via class): 3
total_created (via instance a): 3
total_created (via instance b): 3
t2.members also affected: ['Alice']


## 4. Instance Methods, Class Methods, and Static Methods

**Why we need it:** an ordinary instance method always needs `self` — one already-existing object — which is fine for most behavior, but breaks down in two situations: building an object a different way (there's no object yet, so there's no `self` to use), and writing a small helper that's related to the class but doesn't touch any object's or the class's data at all (using `self` or `cls` for it would be misleading, since it needs neither).

**Use case:**
- **classmethod** — an alternate constructor, e.g. `Rectangle.square(3)` as a shortcut for "make a `Rectangle` where width and height are both 3."
- **staticmethod** — a small check or calculation the class relies on but that doesn't depend on any particular object, e.g. `Rectangle.is_positive(x)`, kept inside the class purely to stay grouped with related code instead of floating loose in the file.

**How to implement:**
- **Instance method** — default, no decorator. First parameter is `self`.
- **Class method** — decorate with `@classmethod`. First parameter is `cls` (the class itself, not an object), which lets it build a new instance with `cls(...)`.
- **Static method** — decorate with `@staticmethod`. No automatic first parameter at all — just the arguments you pass explicitly.

**Quick check:** does the method use `self.something`? → instance method. Does it need to build or reason about the class as a whole? → classmethod. Needs neither? → staticmethod.

In [4]:
class Rectangle:
    unit = "cm"  # class attribute, shared by all rectangles

    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):                       # instance method: uses self (one object's data)
        return self.width * self.height

    @classmethod
    def square(cls, side):                # class method: uses cls (the class), builds a new object
        return cls(side, side)

    @staticmethod
    def is_positive(value):               # static method: uses neither self nor cls
        return value > 0


r = Rectangle(4, 5)
print("area:", r.area())                 # instance method -> called on an object

sq = Rectangle.square(3)                  # class method -> called on the class, creates an object
print("square area:", sq.area())

print(Rectangle.is_positive(10))          # static method -> just a plain helper function
print(Rectangle.is_positive(-2))

area: 20
square area: 9
True
False


## 5. Making Objects Printable: `__str__` and `__repr__`

**Why we need it:** if you print an object without defining anything special, Python shows something like `<__main__.Point object at 0x7f2b1c0a4d90>` — technically correct, useless to a human. Python has no idea what a "readable" version of your object should look like unless you tell it.

**Use case:** anywhere an object needs to be shown to a person (`print(user)`, building a log line, showing an error message) or shown to a developer while debugging (inspecting a variable in a notebook cell, or an object nested inside a list/dict that gets printed).

**How to implement:** define two methods on the class.
- `__str__(self)` → return a friendly, human-readable string. Used by `print(obj)` and `str(obj)`.
- `__repr__(self)` → return a precise, unambiguous string — `repr` is short for **representation**. Ideally it looks like the code you'd type to recreate the object, e.g. `"Point(x=3, y=4)"`. Used by `repr(obj)`, by typing the object's bare name in a notebook cell, and by anything that prints a *collection* of objects (a `list`, `dict`, etc. always uses each element's `__repr__`, never its `__str__`).

**If `__str__` already exists, why do we also need `__repr__`?** Because they're read by different audiences in different places. `__str__` is what you show a user; `__repr__` is what you (or a `list` of your objects) show while debugging, and Python never substitutes one for the other automatically — except in one direction: if you define `__repr__` but skip `__str__`, `print()` falls back to `__repr__`. If you define `__str__` but skip `__repr__`, printing a *single* object works fine, but printing a *list* of them still falls back to the default, useless `<... object at 0x...>` form, because lists never call `__str__` on their elements. That's the concrete case the example below shows — it's also why, if you only write one, `__repr__` is the one worth writing.

In [5]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        return f"({self.x}, {self.y})"

    def __repr__(self):
        return f"Point(x={self.x}, y={self.y})"


class PointNoRepr:
    """Same idea, but only __str__ is defined -- no __repr__."""
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        return f"({self.x}, {self.y})"


p = Point(3, 4)
print(p)          # uses __str__          -> (3, 4)
print(str(p))     # uses __str__          -> (3, 4)
print(repr(p))    # uses __repr__         -> Point(x=3, y=4)
p                  # bare name in a notebook cell -> uses __repr__

points = [Point(1, 1), Point(2, 2)]
print(points)      # a list prints its elements using __repr__, so this looks good

q = PointNoRepr(1, 1)
print(q)                # fine -- print() uses __str__ directly on a single object
print([q, PointNoRepr(2, 2)])   # NOT fine -- a list falls back to the default repr

(3, 4)
(3, 4)
Point(x=3, y=4)
[Point(x=1, y=1), Point(x=2, y=2)]
(1, 1)
[<__main__.PointNoRepr object at 0x77d6c036f8c0>, <__main__.PointNoRepr object at 0x77d6c03e1810>]


## 6. Putting It Together: a small `BankAccount` example

**Why we need it:** each concept above was shown in isolation, but real classes combine all of them at once — a good time to check they actually click together.

**Use case:** a small but realistic class, `BankAccount`, that needs per-account state, behavior that changes that state, configuration shared by every account, and a second way to create an account.

**How it's implemented here:**
- `__init__` and instance attributes (`owner`, `balance`, `account_id`) set up each account's own state.
- `deposit`/`withdraw` are instance methods that change that state.
- `bank_name` and `_next_id` are class attributes — one is shared config, the other hands out a unique id to every new account without any single account needing to know how many others exist.
- `open_with_bonus` is a classmethod offering a second way to create an account, on top of the normal constructor.
- `__str__` makes `print(account)` show something readable, instead of the default `<... object at 0x...>`.

In [6]:
class BankAccount:
    bank_name = "Python National Bank"  # class attribute, shared config
    _next_id = 1                        # class attribute used to auto-generate ids

    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance
        self.account_id = BankAccount._next_id
        BankAccount._next_id += 1

    def deposit(self, amount):
        self.balance += amount
        return self.balance

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("insufficient funds")
        self.balance -= amount
        return self.balance

    @classmethod
    def open_with_bonus(cls, owner, bonus=50):
        """Alternate constructor: opens an account with a signup bonus."""
        return cls(owner, balance=bonus)

    def __str__(self):
        return f"Account #{self.account_id} ({self.owner}) @ {self.bank_name}: {self.balance}"


acc1 = BankAccount("Alice", balance=100)
acc2 = BankAccount.open_with_bonus("Bob")

acc1.deposit(50)
acc2.withdraw(20)

print(acc1)
print(acc2)
print(f"Next id to be assigned: {BankAccount._next_id}")

Account #1 (Alice) @ Python National Bank: 150
Account #2 (Bob) @ Python National Bank: 30
Next id to be assigned: 3
